In [50]:
from pathlib import Path
import re
import unicodedata

import nltk
import pandas as pd
from IPython.display import display
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm

nltk.download("stopwords", quiet=True)

DATA_FOLDER = Path("../../data")
INPUT_PATH = DATA_FOLDER / "translated" / "translated_data.csv"

print(f"Input : {INPUT_PATH.resolve()}")

Input : C:\Works\uts-ir\src\data\translated\translated_data.csv


In [51]:
# Load the dataset
df = pd.read_csv(INPUT_PATH)
df.head()


,id,title,title_language,title_translated,abstract,abstract_language,abstract_translated
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,id,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,id,penelitian ini dilatarbelakangi oleh perkemban...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,id,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,id,penelitian ini membahas hubungan antara penggu...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,id,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,id,penelitian ini membahas hubungan antara keters...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,id,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,id,masalah yang menjadi kajian dalam penelitian a...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,id,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,id,fenomena pesatnya perkembangan ilmu pengetahua...


In [52]:
df.shape

(5890, 7)

In [53]:
# Check Duplicate title_translated and abstract_translated

df.duplicated(subset=["title_translated", "abstract_translated"]).sum()

# Display the duplicated rows
duplicated_rows = df[df.duplicated(subset=["title_translated", "abstract_translated"], keep=False)].sort_values(by=["title_translated", "abstract_translated"])
duplicated_rows

,id,title,title_language,title_translated,abstract,abstract_language,abstract_translated
2089,2723-6234-27,awareness and perceptions of artificial intell...,en,Kesadaran dan persepsi tentang kecerdasan buat...,permasalahan: penemuan kecerdasan buatan (arti...,id,permasalahan: penemuan kecerdasan buatan (arti...
2103,2723-6234-41,awareness and perceptions of artificial intell...,en,Kesadaran dan persepsi tentang kecerdasan buat...,permasalahan: penemuan kecerdasan buatan (arti...,id,permasalahan: penemuan kecerdasan buatan (arti...
2091,2723-6234-29,digital literacy for state apparatus in suppor...,en,Literasi digital bagi aparat negara dalam mend...,latar belakang: penelitian ini mengangkat perm...,id,latar belakang: penelitian ini mengangkat perm...
2105,2723-6234-43,digital literacy for state apparatus in suppor...,en,Literasi digital bagi aparat negara dalam mend...,latar belakang: penelitian ini mengangkat perm...,id,latar belakang: penelitian ini mengangkat perm...
2092,2723-6234-30,"health literacy of children among parents in ""...",en,Literasi kesehatan anak di kalangan orang tua ...,latar belakang: anak usia dini masih belum mem...,id,latar belakang: anak usia dini masih belum mem...
2106,2723-6234-44,"health literacy of children among parents in ""...",en,Literasi kesehatan anak di kalangan orang tua ...,latar belakang: anak usia dini masih belum mem...,id,latar belakang: anak usia dini masih belum mem...
2094,2723-6234-32,civil servants' literacy regarding their main ...,en,Literasi pegawai negeri sipil mengenai tugas d...,latar belakang/permasalahan: pelaksanaan penga...,id,latar belakang/permasalahan: pelaksanaan penga...
2108,2723-6234-46,civil servants' literacy regarding their main ...,en,Literasi pegawai negeri sipil mengenai tugas d...,latar belakang/permasalahan: pelaksanaan penga...,id,latar belakang/permasalahan: pelaksanaan penga...
2407,2477-0361-166,analisis sitiran skripsi program studi ilmu pe...,id,analisis sitiran skripsi program studi ilmu pe...,pendahuluan penelitian ini mengkaji perilaku p...,id,pendahuluan penelitian ini mengkaji perilaku p...
2414,2477-0361-173,analisis sitiran skripsi program studi ilmu pe...,id,analisis sitiran skripsi program studi ilmu pe...,pendahuluan penelitian ini mengkaji perilaku p...,id,pendahuluan penelitian ini mengkaji perilaku p...


In [54]:
# Check #ERROR! value in title_translated and abstract_translated columns
duplicated_rows = df[
  df["title_translated"].astype(str).str.contains("#ERROR!", na=False)
  | df["abstract_translated"].astype(str).str.contains("#ERROR!", na=False)
].copy()

duplicated_rows


,id,title,title_language,title_translated,abstract,abstract_language,abstract_translated


In [55]:
# Drop duplicates based on title_translated and abstract_translated, keeping the first occurrence
df = df.drop_duplicates(subset=["title_translated", "abstract_translated"], keep="first").reset_index(drop=True)

df.duplicated(subset=["title_translated", "abstract_translated"]).sum()

np.int64(0)

In [56]:
df.shape

(5871, 7)

In [57]:
# Export the cleaned dataset to a new CSV file

OUTPUT_CLEANED_TRANSLATED_DUPLICATED_PATH = DATA_FOLDER / "translated" / "cleaned_translated_data.csv"

df.to_csv(OUTPUT_CLEANED_TRANSLATED_DUPLICATED_PATH, index=False)

print(f"Cleaned dataset exported to: {OUTPUT_CLEANED_TRANSLATED_DUPLICATED_PATH.resolve()}")

Cleaned dataset exported to: C:\Works\uts-ir\src\data\translated\cleaned_translated_data.csv


# Normalizing Text


In [58]:
def normalize_text(text):
    # Handle missing values
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize Unicode without forcing text into ASCII
    # e.g., "café" remains "café" instead of being converted to "cafe"
    text = unicodedata.normalize("NFC", text)

    # Case folding
    text = text.casefold()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)

    # Remove digits
    text = re.sub(r"\d+", " ", text)

    # Remove punctuation and special characters,
    # while preserving letters and whitespace
    # e.g., "Hello, World!" becomes "Hello World"
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # Replace underscores with spaces
    text = text.replace("_", " ")

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [59]:
df["combined_text"] = (
    df[["title_translated", "abstract_translated"]]
    .fillna("")
    .astype(str)
    .agg(lambda row: " ".join(row), axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["normalized_text"] = df["combined_text"].apply(normalize_text)

df[["title_translated", "abstract_translated", "combined_text", "normalized_text"]].sample(20, random_state=42)

,title_translated,abstract_translated,combined_text,normalized_text
2715,peran pustakawan dalam pengenalan literasi dig...,dengan perkembangan teknologi yang sangat pesa...,peran pustakawan dalam pengenalan literasi dig...,peran pustakawan dalam pengenalan literasi dig...
586,pengelolaan arsip desa kabupaten rembang dalam...,arsip yang tercipta dalam kegiatan pemerintaha...,pengelolaan arsip desa kabupaten rembang dalam...,pengelolaan arsip desa kabupaten rembang dalam...
199,penentuan tajuk entri utama pada proses katalo...,permasalahan. tajuk entri utama merupakan sala...,penentuan tajuk entri utama pada proses katalo...,penentuan tajuk entri utama pada proses katalo...
439,tanggapan pengguna perpustakaan terhadap pengg...,penelitian ini dilakukan bertujuan agar menget...,tanggapan pengguna perpustakaan terhadap pengg...,tanggapan pengguna perpustakaan terhadap pengg...
1268,filsafat eksistensialisme dan relevansinya den...,maraknya keberadaan perpustakaan komunitas dis...,filsafat eksistensialisme dan relevansinya den...,filsafat eksistensialisme dan relevansinya den...
2609,digitalisasi arsip perusahaan e-commerce (blib...,setiap organisasi atau perusahaan tidak terlep...,digitalisasi arsip perusahaan e-commerce (blib...,digitalisasi arsip perusahaan e commerce blibl...
3114,implementasi budaya riset sebagai mediapengemb...,taman bacaan masyarakat (tbm) merupakan salah ...,implementasi budaya riset sebagai mediapengemb...,implementasi budaya riset sebagai mediapengemb...
5691,penyusunan direktori insan ombudsman republik ...,pendahuluan.tulisan ini mengangkat tema tentan...,penyusunan direktori insan ombudsman republik ...,penyusunan direktori insan ombudsman republik ...
907,Studi bibliometrik tentang tesis mahasiswa di ...,bibliometrika merupakan kajian keilmuan yang s...,Studi bibliometrik tentang tesis mahasiswa di ...,studi bibliometrik tentang tesis mahasiswa di ...
5378,evaluasi layanan administrasi online perpustak...,evaluasi layanan administrasi online perpustak...,evaluasi layanan administrasi online perpustak...,evaluasi layanan administrasi online perpustak...


# Tokenization


In [60]:
df["tokenized_text"] = df["normalized_text"].apply(lambda x: x.split())

df[["normalized_text", "tokenized_text"]].sample(20, random_state=42)

,normalized_text,tokenized_text
2715,peran pustakawan dalam pengenalan literasi dig...,"[peran, pustakawan, dalam, pengenalan, literas..."
586,pengelolaan arsip desa kabupaten rembang dalam...,"[pengelolaan, arsip, desa, kabupaten, rembang,..."
199,penentuan tajuk entri utama pada proses katalo...,"[penentuan, tajuk, entri, utama, pada, proses,..."
439,tanggapan pengguna perpustakaan terhadap pengg...,"[tanggapan, pengguna, perpustakaan, terhadap, ..."
1268,filsafat eksistensialisme dan relevansinya den...,"[filsafat, eksistensialisme, dan, relevansinya..."
2609,digitalisasi arsip perusahaan e commerce blibl...,"[digitalisasi, arsip, perusahaan, e, commerce,..."
3114,implementasi budaya riset sebagai mediapengemb...,"[implementasi, budaya, riset, sebagai, mediape..."
5691,penyusunan direktori insan ombudsman republik ...,"[penyusunan, direktori, insan, ombudsman, repu..."
907,studi bibliometrik tentang tesis mahasiswa di ...,"[studi, bibliometrik, tentang, tesis, mahasisw..."
5378,evaluasi layanan administrasi online perpustak...,"[evaluasi, layanan, administrasi, online, perp..."


In [61]:
stopwords_id = set(stopwords.words("indonesian"))

def remove_stopwords(tokens):
    # Token with length greater than 1 and not in stopwords_id
    return [token for token in tokens if token not in stopwords_id and len(token) > 1]

df["tokenized_no_stopwords_text"] = df["tokenized_text"].map(remove_stopwords)
display(df[["tokenized_text", "tokenized_no_stopwords_text"]].sample(20, random_state=42))
print(f"Jumlah stopword dalam daftar: {len(stopwords_id):,}")

,tokenized_text,tokenized_no_stopwords_text
2715,"[peran, pustakawan, dalam, pengenalan, literas...","[peran, pustakawan, pengenalan, literasi, digi..."
586,"[pengelolaan, arsip, desa, kabupaten, rembang,...","[pengelolaan, arsip, desa, kabupaten, rembang,..."
199,"[penentuan, tajuk, entri, utama, pada, proses,...","[penentuan, tajuk, entri, utama, proses, katal..."
439,"[tanggapan, pengguna, perpustakaan, terhadap, ...","[tanggapan, pengguna, perpustakaan, penggunaan..."
1268,"[filsafat, eksistensialisme, dan, relevansinya...","[filsafat, eksistensialisme, relevansinya, per..."
2609,"[digitalisasi, arsip, perusahaan, e, commerce,...","[digitalisasi, arsip, perusahaan, commerce, bl..."
3114,"[implementasi, budaya, riset, sebagai, mediape...","[implementasi, budaya, riset, mediapengembanga..."
5691,"[penyusunan, direktori, insan, ombudsman, repu...","[penyusunan, direktori, insan, ombudsman, repu..."
907,"[studi, bibliometrik, tentang, tesis, mahasisw...","[studi, bibliometrik, tesis, mahasiswa, jurusa..."
5378,"[evaluasi, layanan, administrasi, online, perp...","[evaluasi, layanan, administrasi, online, perp..."


Jumlah stopword dalam daftar: 757


# Stemming


In [62]:
stemmer = StemmerFactory().create_stemmer()

def stem_tokens(tokens):
    return [stemmer.stem(token) for token in tokens]

df["tokenized_stemmed_text"] = [
    stem_tokens(tokens)
    for tokens in tqdm(
        df["tokenized_no_stopwords_text"],
        desc="Stemming combined text"
    )
]

df["stemmed_text"] = df["tokenized_stemmed_text"].apply(" ".join)


Stemming combined text: 100%|██████████| 5871/5871 [31:03<00:00,  3.15it/s]  


In [65]:
# Periksa hasilnya
df[[
    "tokenized_no_stopwords_text",
    "tokenized_stemmed_text",
    "stemmed_text"
]].sample(min(20, len(df)), random_state=42)

,tokenized_no_stopwords_text,tokenized_stemmed_text,stemmed_text
2715,"[peran, pustakawan, pengenalan, literasi, digi...","[peran, pustakawan, kenal, literasi, digital, ...",peran pustakawan kenal literasi digital siswa ...
586,"[pengelolaan, arsip, desa, kabupaten, rembang,...","[kelola, arsip, desa, kabupaten, rembang, tunj...",kelola arsip desa kabupaten rembang tunjang pe...
199,"[penentuan, tajuk, entri, utama, proses, katal...","[tentu, tajuk, entri, utama, proses, katalogis...",tentu tajuk entri utama proses katalogisasi ba...
439,"[tanggapan, pengguna, perpustakaan, penggunaan...","[tanggap, guna, pustaka, guna, katalog, online...",tanggap guna pustaka guna katalog online opac ...
1268,"[filsafat, eksistensialisme, relevansinya, per...","[filsafat, eksistensialisme, relevansi, pustak...",filsafat eksistensialisme relevansi pustaka ba...
2609,"[digitalisasi, arsip, perusahaan, commerce, bl...","[digitalisasi, arsip, usaha, commerce, blibli,...",digitalisasi arsip usaha commerce blibli softw...
3114,"[implementasi, budaya, riset, mediapengembanga...","[implementasi, budaya, riset, mediapengembanga...",implementasi budaya riset mediapengembangan po...
5691,"[penyusunan, direktori, insan, ombudsman, repu...","[susun, rektor, insan, ombudsman, republik, in...",susun rektor insan ombudsman republik indonesi...
907,"[studi, bibliometrik, tesis, mahasiswa, jurusa...","[studi, bibliometrik, tesis, mahasiswa, jurus,...",studi bibliometrik tesis mahasiswa jurus didik...
5378,"[evaluasi, layanan, administrasi, online, perp...","[evaluasi, layan, administrasi, online, pustak...",evaluasi layan administrasi online pustaka fmi...


In [67]:
# Export the processed DataFrame to a new CSV file
df_processed = pd.DataFrame({
    "id": df["id"],
    "title": df["title"],
    "abstract": df["abstract"],
    "title_translated": df["title_translated"],
    "abstract_translated": df["abstract_translated"],
    "processed_token_text": df["tokenized_stemmed_text"],
    "processed_text": df["stemmed_text"],
})

df_processed.head()

,id,title,abstract,title_translated,abstract_translated,processed_token_text,processed_text
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,"[hubung, promosi, pustaka, majalah, dinding, m...",hubung promosi pustaka majalah dinding minat k...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,"[hubung, guna, search, features, journals, pen...",hubung guna search features journals penuh but...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,"[hubung, sedia, fasilitas, pustaka, minat, kun...",hubung sedia fasilitas pustaka minat kunjung s...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,"[pengaruh, kerja, layan, prima, center, of, in...",pengaruh kerja layan prima center of informati...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,"[kontribusi, literasi, informasi, mahasiswa, p...",kontribusi literasi informasi mahasiswa proses...


In [68]:
OUTPUT_FOLDER_PATH = DATA_FOLDER / "processed"
OUTPUT_FOLDER_PATH.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_FOLDER_PATH / "processed_data.csv"
df_processed.to_csv(OUTPUT_PATH, index=False)

print(f"Processed data saved to: {OUTPUT_PATH.resolve()}")

Processed data saved to: C:\Works\uts-ir\src\data\processed\processed_data.csv
